In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import numpy as np
import os
import pandas as pd
from collections import defaultdict
pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

In [3]:
from utils_shared_hanzi_shorts import (
    data_settings_by_char,
    enhance_data_settings, get_filtered_words, load_video_configs,
    get_character_counts,
    stitch_audios, draw_vocab_list_whole_image, 
    create_video_with_highlights, create_video_without_highlights
)
from utils_shorts import (
    create_directories
)
from utils_data import (load_raw_data, check_dups)

In [9]:
truly_load_data = False
df_all_vocab = load_raw_data(truly_load_data=truly_load_data)
df_dups = check_dups(df_all_vocab)
print(df_all_vocab.shape)
print(f'# duplicate vocab: {len(df_dups)}')
df_all_vocab.head(3)

(7665, 38)
# duplicate vocab: 0


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,1,房贷,fáng dài,mortgage,word,1,life,NaN,Finance & Economy,NaN,借贷与负债,MISSING,1.0,1.0,2.0,1.0,房子,house,贷款,loan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他每个月都要还房贷,Ta měi gè yuè dōu yào huán fángdài,He has to pay his mortgage every month,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,2,白天,bái tiān,daytime,word,2,time,NaN,Time,NaN,时间段 / 时长,1.0,2.0,1.0,1.0,1.0,白,white,天,day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,白天很热晚上比较凉快,Báitiān hěn rè wǎnshàng bǐjiào liángkuai,It is hot in the daytime and cooler at night,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,3,组成,zǔ chéng,to form;make up,word,3,general,NaN,Language & Expression,NaN,逻辑与结构,MISSING,5.0,5.0,5.0,3.0,组,set,成,become,NaN,NaN,NaN,NaN,NaN,NaN,NaN,水是由氢和氧组成的,Shuǐ shì yóu qīng hé yǎng zǔchéng de,Water is made up of hydrogen and oxygen,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


# Settings

In [11]:
audio_settings = {
    'voice_name_zh': 'zh-CN-XiaoxiaoNeural',
    'audio_plan': 'ctitle_c2word',
    'pause_ms_beginning': 150,
    'pause_ms_within_word': 200,
    'pause_ms_between': 500,
}

# '天', '金', '物', '语', '时', '红', '公', '发', '自', '一', '山', '头', '花', '外', '手', '生', 
chosen_char= '生'

video_configs = load_video_configs()

In [15]:
# from edge_tts import list_voices, Communicate
# def single_tts_call(text, voice_name, output_file_name):
#   communicate = Communicate(text, voice_name)
#   communicate.save_sync(output_file_name)

# # Make dict of lists of words for text to speech
# # Or just directly do tts for these lists
# for char, settings in data_settings_by_char.items():
#     if char not in ['地', '水', '动']:
#         print(f"Processing character: {char}")
#         data_settings = data_settings_by_char[char]
#         data_settings = enhance_data_settings(data_settings)
#         create_directories(data_settings)
#         df_filt = get_filtered_words(df_all_vocab, data_settings)
#         words_to_tts = [data_settings['shared_char']] + df_filt['chinese'].values.tolist()
#         for word in words_to_tts:
#             output_file_name = f'output/shared_char_shorts/{char}/audio_files/{word}.mp3'
#             print(char, word)
#             if not os.path.exists(output_file_name):
#                 single_tts_call(word, 'zh-CN-XiaoxiaoNeural', output_file_name)

In [ ]:
for char, settings in data_settings_by_char.items():
    if char not in ['地', '水', '动']:
        data_settings = data_settings_by_char[char]
        data_settings = enhance_data_settings(data_settings)
        df_filt = get_filtered_words(df_all_vocab, data_settings)

        data_settings['n_words_total'] = len(df_filt)
        data_settings['n_parts'] = int(np.ceil(len(df_filt) / data_settings['n_words_per_video']))
        print(f"{char}: {len(df_filt)} words, {data_settings['n_parts']} parts")

        # Decrease spacing if necessary
        if data_settings['n_words_per_video'] == 11:
            video_configs['words_settings']['spacing'] = 48
        elif data_settings['n_words_per_video'] == 12:
            video_configs['words_settings']['spacing'] = 40

        for current_part in range(1, data_settings['n_parts'] + 1):
            # Determine vocabulary in current part
            data_settings['current_part'] = current_part
            start_index = (current_part - 1) * data_settings['n_words_per_video']
            end_index = start_index + data_settings['n_words_per_video'] - 1
            data_settings['current_part_index_range'] = (start_index, end_index)
            print(f"Processing part {current_part}/{data_settings['n_parts']} with index range {data_settings['current_part_index_range']}")
            df_filt_currentpart = df_filt[
                (df_filt.index >= data_settings['current_part_index_range'][0]) &
                (df_filt.index <= data_settings['current_part_index_range'][1])
            ].reset_index(drop=True)
            df_durations = stitch_audios(audio_settings, data_settings, df_filt_currentpart['chinese'].values.tolist())
            no_hl_img_file_path = draw_vocab_list_whole_image(video_configs, data_settings, df_filt_currentpart)
            create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path)
            create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs)


/Users/scott/Development/my_mandarin_database/utils_shared_hanzi_shorts.py:606: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_filtered = df_filtered.replace(data_settings['text_replacements'])


天: 20 words, 2 parts
Processing part 1/2 with index range (0, 9)
Audio duration: 34.7s
MoviePy - Building video output/shared_char_shorts/天/天_no_highlights_part1.mp4.
MoviePy - Writing audio in 天_no_highlights_part1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/天/天_no_highlights_part1.mp4



MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/天/天_no_highlights_part1.mp4
MoviePy - Building video output/shared_char_shorts/天/天_part1.mp4.
MoviePy - Writing audio in 天_part1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/天/天_part1.mp4



...

MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/生/生_no_highlights_part2.mp4



MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/生/生_no_highlights_part2.mp4
MoviePy - Building video output/shared_char_shorts/生/生_part2.mp4.
MoviePy - Writing audio in 生_part2TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/生/生_part2.mp4



frame_index: 100%|█████████▉| 1044/1045 [00:18<00:00, 58.23it/s, now=None]/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file output/shared_char_shorts/生/生_no_highlights_part2.mp4, 2764800 bytes wanted but 0 bytes read at frame index 1044 (out of a total 1044 frames), at time 43.50/43.52 sec. Using the last valid frame instead.
  warnings.warn(
                                                                          

MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/生/生_part2.mp4


In [5]:
data_settings = data_settings_by_char[chosen_char]
data_settings = enhance_data_settings(data_settings)
create_directories(data_settings)
data_settings

{'shared_char': '生',
 'char_pinyin': 'shēng',
 'char_english': 'life',
 'max_priority': 6,
 'min_adu': 3,
 'min_per': 3,
 'types_allowed': ['word',
  'prefix',
  'suffix',
  'abbreviation',
  'multi_word',
  'verb_ending',
  'proper noun'],
 'sort_cols': ['priority', 'cat_v3', 'pinyin'],
 'sort_ascending': [True, True, True],
 'words_rmv': ['花生酱',
  '轻奢生活',
  '生育率',
  '独生女',
  '独生子',
  '博士生导师',
  '咖啡生产地',
  '出生率',
  '低生育率',
  '招生',
  '精神卫生科',
  '生效',
  '生产者名称'],
 'n_words_per_video': 12,
 'current_part': 1,
 'text_replacements': {'to produce;production': 'production',
  'lifestyle;way of life': 'lifestyle',
  'business (how doing)': 'business',
  'physiological;physical': 'physiological'},
 'output_path_base': 'output/shared_char_shorts/',
 'output_path': 'output/shared_char_shorts/生',
 'output_path_audio': 'output/shared_char_shorts/生/audio_files',
 'output_path_images': 'output/shared_char_shorts/生/images'}

# Load data

In [7]:
previous_video_chars = ['口', '电', '车', '色', '子', '海'] + list(data_settings_by_char.keys())
df_character_appearances = get_character_counts(df_all_vocab, data_settings)
df_character_appearances = df_character_appearances[~df_character_appearances['character'].isin(previous_video_chars)]
print(len(df_all_vocab), len(df_character_appearances))
df_character_appearances['character'].value_counts().head(30)

7669 11170


character
大    80
人    80
机    48
科    48
心    45
不    39
中    38
家    37
小    37
国    36
自    36
时    33
发    33
学    31
公    31
红    31
的    30
平    30
语    30
物    30
金    29
天    29
西    29
上    29
行    29
气    29
猫    29
体    28
可    28
点    28
Name: count, dtype: int64

In [20]:
data_settings['shared_char'] = '天'
df_filt = get_filtered_words(df_all_vocab, data_settings)
print([data_settings['shared_char']] + df_filt['chinese'].values.tolist())
print({k:k for k in df_filt['english'].values.tolist()})
print(len(df_filt))
df_filt.head(40)

['天', '天气预报', '天堂', '白天', '第二天', '平均每天', '这几天', '天空', '天安门广场', '天花板', '天桥', '天才', '天真', '摩天轮', '天猫', '天津', '天鹅', '天文', '摩天楼', '天线', '天然', '天然气', '天妇罗', '先天', '天坛', '天蓝', '天主教', '天平', '天津市', '天王星']
{'weather forecast': 'weather forecast', 'heaven;paradise': 'heaven;paradise', 'daytime': 'daytime', 'the next day': 'the next day', 'Average daily': 'Average daily', 'These days': 'These days', 'sky': 'sky', 'Tiananmen Square': 'Tiananmen Square', 'ceiling': 'ceiling', 'pedestrian bridge': 'pedestrian bridge', 'genius': 'genius', 'innocent;naive': 'innocent;naive', 'ferris wheel': 'ferris wheel', 'Chinese Amazon (less variety)': 'Chinese Amazon (less variety)', 'Tianjin': 'Tianjin', 'swan': 'swan', 'astronomy': 'astronomy', 'skyscraper': 'skyscraper', 'antenna': 'antenna', 'natural': 'natural', 'natural gas': 'natural gas', 'tempura': 'tempura', 'innate': 'innate', 'Temple of Heaven (Beijing), historic site for imperial ceremonies': 'Temple of Heaven (Beijing), historic site for imperial cer

/Users/scott/Development/my_mandarin_database/utils_shared_hanzi_shorts.py:567: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  'fill': video_configs['words_settings']['fill']['english'],


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,782,天气预报,tiān qì yù bào,weather forecast,multi_word,1,travel,NaN,Weather & Nature,NaN,天气现象,MISSING,4.0,2.0,2.0,3.0,天气,weather,预期,to expect,报,report,NaN,NaN,NaN,NaN,NaN,我看了今天的天气预报,Wǒ kàn le jīntiān de tiānqì yùbào,I checked today's weather forecast,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,1040,天堂,tiān táng,heaven;paradise,word,2,general,NaN,Culture & Society,NaN,宗教与信仰,6.0,2.0,1.0,2.0,2.0,天,sky,堂,hall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他觉得海边是天堂,Tā juéde hǎibiān shì tiāntáng,He thinks the beach is paradise,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,2,白天,bái tiān,daytime,word,2,time,NaN,Time,NaN,时间段 / 时长,1.0,2.0,1.0,1.0,1.0,白,white,天,day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,白天很热晚上比较凉快,Báitiān hěn rè wǎnshàng bǐjiào liángkuai,It is hot in the daytime and cooler at night,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
3,3738,第二天,dì èr tiān,the next day,multi_word,2,time,NaN,Time,NaN,时间顺序与先后,MISSING,2.0,1.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我们第二天再出发,Wǒmen dì èr tiān zài chūfā,We will set out the next day,2025-06-08,daily add,NaN,NaN,5.0,5.0,5.0,NaN
4,3389,平均每天,píng jūn měi tiān,Average daily,multi_word,2,amount,degree,Time,NaN,频率与习惯,MISSING,2.0,1.0,5.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我平均每天跑步三公里,Wǒ píngjūn měitiān pǎobù sān gōnglǐ,I run three kilometers every day on average,2025-02-10,ltl,NaN,NaN,5.0,5.0,5.0,NaN
5,3326,这几天,zhè jǐ tiān,These days,multi_word,2,time,NaN,Time,NaN,现在与当前,MISSING,3.0,2.0,5.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这几天很忙,Zhè jǐ tiān hěn máng,These past few days have been busy,2025-02-10,ltl,NaN,NaN,5.0,5.0,5.0,NaN
6,5928,天空,tiān kōng,sky,word,2,NaN,NaN,Weather & Nature,NaN,天体与天空现象,3.0,5.0,5.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,今天的天空很蓝,jīn tiān de tiān kōng hěn lán,The sky is very blue today,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
7,6795,天安门广场,tiān ān mén guǎng chǎng,Tiananmen Square,proper noun,2,NaN,NaN,my_china_spots,NaN,NaN,MISSING,5.0,5.0,5.0,4.0,天空,sky,安全,safe,门口,gate,广场,square,NaN,NaN,NaN,清晨的天安门广场非常庄严。,qīng chén de tiān ān mén guǎng chǎng fēi cháng...,Tiananmen Square is very solemn in the early m...,2025-12-30,china places,NaN,NaN,5.0,5.0,5.0,NaN
8,5605,天花板,tiān huā bǎn,ceiling,word,3,house,NaN,Architecture & Infrastructure,NaN,NaN,MISSING,5.0,5.0,5.0,2.0,天,sky,花,flower,板,board,NaN,NaN,NaN,NaN,NaN,房间的天花板上装着漂亮的灯,fáng jiān de tiān huā bǎn shàng zhuāng zhe pià...,A beautiful lamp is installed on the ceiling,2025-10-24,daily add,NaN,NaN,5.0,5.0,5.0,NaN
9,6117,天桥,tiān qiáo,pedestrian bridge,word,3,NaN,NaN,Architecture & Infrastructure,NaN,NaN,MISSING,5.0,5.0,5.0,4.0,天空,sky,桥,bridge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我们在天桥上等你,wǒ men zài tiān qiáo shàng děng nǐ,We’ll wait for you on the pedestrian bridge,2025-11-11,daily add,NaN,NaN,5.0,5.0,5.0,NaN


# Create video for each part

In [9]:
print('Cut vocab into parts')
data_settings['n_words_total'] = len(df_filt)
data_settings['n_parts'] = int(np.ceil(len(df_filt) / data_settings['n_words_per_video']))

for current_part in range(1, data_settings['n_parts'] + 1):
    # Determine vocabulary in current part
    data_settings['current_part'] = current_part
    start_index = (current_part - 1) * data_settings['n_words_per_video']
    end_index = start_index + data_settings['n_words_per_video'] - 1
    data_settings['current_part_index_range'] = (start_index, end_index)
    print(f"Processing part {current_part}/{data_settings['n_parts']} with index range {data_settings['current_part_index_range']}")
    df_filt_currentpart = df_filt[
        (df_filt.index >= data_settings['current_part_index_range'][0]) &
        (df_filt.index <= data_settings['current_part_index_range'][1])
    ].reset_index(drop=True)

    print('Making audio')
    df_durations = stitch_audios(audio_settings, data_settings, df_filt_currentpart['chinese'].values.tolist())
    print('Making image')
    no_hl_img_file_path = draw_vocab_list_whole_image(video_configs, data_settings, df_filt_currentpart)
    print('Making video without highlights')
    create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path)
    print('Making video with highlights')
    create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs)

Cut vocab into parts
Processing part 1/2 with index range (0, 11)
Making audio


/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


FileNotFoundError: [Errno 2] No such file or directory: 'output/shared_char_shorts/生/audio_files/生.mp3'